# STRATOVANE thermal model

Literature AAO optics are **not** measured STRATOVANE properties. This notebook asks whether a treated aluminum enclosure is predicted to beat bare aluminum and a white reference at 20 km — and will report a negative answer if that is what the physics says.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()))

from config import SimulationConfig, DEFAULT_ALTITUDES_M
from enclosure import solve_steady, comparison_table
from materials import load_materials
from atmosphere import atmosphere_at
from spectral import planck_sigma_check
from validation import diaz_lobo_reproduction
import pandas as pd

In [ ]:
print('Planck check', planck_sigma_check(300))
a20 = atmosphere_at(20000)
print('US76 20 km T,P,rho', a20.temperature_k, a20.pressure_pa, a20.density_kg_m3)
print('T_air is not T_sky')

In [ ]:
print(diaz_lobo_reproduction())

In [ ]:
cfg = SimulationConfig()
cfg.environment.radiation_mode = 'effective_sky'  # faster in the notebook; use spectral in CLI mode 4/10
cfg.environment.irradiance_direct_w_m2 = 900.0  # USER SCENARIO, not a universal solar constant
mats = load_materials()
ids = {
    'bare_aluminum': cfg.bare_material_id,
    'white_reference': cfg.white_material_id,
    'stratovane_aao_literature': cfg.stratovane_material_id,
}
rows = []
for z in DEFAULT_ALTITUDES_M:
    for label, mid in ids.items():
        s = solve_steady(cfg, mats[mid], altitude_m=z)
        rows.extend(comparison_table({label: s}))
pd.DataFrame(rows)